# Phase 10 - Revenue-at-Risk Analysis

## Business Problem

Predicting churn alone is not sufficient.

The business also needs to understand which customers contribute the highest financial risk if they churn.

## Objective

Estimate expected revenue at risk by combining customer revenue with churn probability.

## Business Value

Helps marketing teams prioritize retention campaigns based on financial impact rather than treating every customer equally.

In [1]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

features = pd.read_csv(
"/content/drive/MyDrive/Revenue_Intelligence_Project/Data/model_dataset.csv"
)

model_data = pd.read_csv(
"/content/drive/MyDrive/Revenue_Intelligence_Project/Data/model_ready_dataset.csv"
)

Mounted at /content/drive


In [2]:
import joblib

# ==========================================
# Load Trained Model
# ==========================================

model = joblib.load(
    "/content/drive/MyDrive/Revenue_Intelligence_Project/Data/xgboost_churn_model.pkl"
)

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [6]:
X_prediction = model_data[
    [
        "Frequency",
        "Monetary",
        "AverageOrderValue",
        "CustomerLifetime",
        "AverageDaysBetweenOrders",
        "CustomerAgeEncoded"
    ]
].copy()

print(X_prediction.shape)

X_prediction.head()

(5881, 6)


,Frequency,Monetary,AverageOrderValue,CustomerLifetime,AverageDaysBetweenOrders,CustomerAgeEncoded
0,8,2433.28,304.160000,581,82.714286,0
1,57,29018.06,509.088772,735,12.714286,0
2,2,613.08,306.540000,290,290.000000,1
3,145,580987.04,4006.807172,738,4.868056,0
4,52,22950.22,441.350385,735,13.901961,0


In [7]:
features["ChurnProbability"] = (
    model.predict_proba(X_prediction)[:,1]
)

In [8]:
features["ChurnProbability"] = model.predict_proba(X_prediction)[:,1]

In [9]:
features[
    [
        "Customer ID",
        "Monetary",
        "ChurnProbability"
    ]
].head()

,Customer ID,Monetary,ChurnProbability
0,13085.0,2433.28,0.224913
1,13078.0,29018.06,0.000830
2,15362.0,613.08,0.744811
3,18102.0,580987.04,0.000648
4,12682.0,22950.22,0.000845


In [10]:
# ==========================================
# Calculate Revenue At Risk
# ==========================================

features["RevenueAtRisk"] = (
    features["Monetary"] *
    features["ChurnProbability"]
)

features[
    [
        "Customer ID",
        "Monetary",
        "ChurnProbability",
        "RevenueAtRisk"
    ]
].head()

,Customer ID,Monetary,ChurnProbability,RevenueAtRisk
0,13085.0,2433.28,0.224913,547.275655
1,13078.0,29018.06,0.000830,24.089475
2,15362.0,613.08,0.744811,456.628837
3,18102.0,580987.04,0.000648,376.553960
4,12682.0,22950.22,0.000845,19.400675


In [11]:
# ==========================================
# Assign Priority
# ==========================================

def priority(prob):

    if prob >= 0.80:
        return "Critical"

    elif prob >= 0.60:
        return "High"

    elif prob >= 0.40:
        return "Medium"

    else:
        return "Low"

features["Priority"] = (
    features["ChurnProbability"]
    .apply(priority)
)

features[
    [
        "Customer ID",
        "ChurnProbability",
        "Priority"
    ]
].head()

,Customer ID,ChurnProbability,Priority
0,13085.0,0.224913,Low
1,13078.0,0.000830,Low
2,15362.0,0.744811,High
3,18102.0,0.000648,Low
4,12682.0,0.000845,Low


In [12]:
revenue_risk = features.sort_values(
    by=["RevenueAtRisk", "ChurnProbability"],
    ascending=[False, False]
)

revenue_risk.head()

,Customer ID,Recency,Frequency,Monetary,Segment,AverageOrderValue,CustomerLifetime,AverageDaysBetweenOrders,CustomerAgeCategory,Churn,ChurnProbability,RevenueAtRisk,Priority
4894,16446.0,1,2,168472.50,Potential Loyalists,84236.250000,204,204.000000,Growing,0,0.675477,113799.353790,High
3513,17450.0,8,51,244784.25,Champions,4799.691176,429,8.300000,Established,0,0.271379,66429.208672,Low
747,12346.0,326,12,77556.46,At Risk,6463.038333,400,35.909091,Established,1,0.576094,44679.818320,Medium
1838,16754.0,372,29,65500.07,At Risk,2258.623103,269,9.285714,Growing,1,0.601590,39404.181749,High
2963,12415.0,24,28,144458.37,Champions,5159.227500,503,18.148148,Established,0,0.250552,36194.320405,Low


In [13]:
revenue_risk[
    [
        "Customer ID",
        "Segment",
        "Monetary",
        "ChurnProbability",
        "RevenueAtRisk",
        "Priority"
    ]
].head(20)

,Customer ID,Segment,Monetary,ChurnProbability,RevenueAtRisk,Priority
4894,16446.0,Potential Loyalists,168472.50,0.675477,113799.353790,High
3513,17450.0,Champions,244784.25,0.271379,66429.208672,Low
747,12346.0,At Risk,77556.46,0.576094,44679.818320,Medium
1838,16754.0,At Risk,65500.07,0.601590,39404.181749,High
2963,12415.0,Champions,144458.37,0.250552,36194.320405,Low
4360,15749.0,At Risk,44534.30,0.703359,31323.609026,High
5244,14096.0,Champions,65164.79,0.467368,30455.910821,Medium
353,17850.0,At Risk,51208.87,0.591284,30278.962947,Medium
4977,15098.0,Need Attention,39916.50,0.698835,27895.040750,High
794,13902.0,At Risk,34095.26,0.690306,23536.146560,High


In [14]:
summary = revenue_risk.groupby("Priority").agg(
    {
        "Customer ID": "count",
        "Monetary": "sum",
        "RevenueAtRisk": "sum"
    }
)

summary

,Customer ID,Monetary,RevenueAtRisk
Priority,,,
Critical,603,1.397893e+05,1.144686e+05
High,2221,1.755536e+06,1.221701e+06
Low,1888,1.323540e+07,1.103782e+06
Medium,1169,2.244080e+06,1.116271e+06


In [15]:
revenue_risk.to_csv(
    "/content/drive/MyDrive/Revenue_Intelligence_Project/Data/revenue_at_risk.csv",
    index=False
)

print("Revenue-at-Risk dataset saved successfully!")

Revenue-at-Risk dataset saved successfully!


In [16]:
# ==========================================
# Step 10 - Revenue Category (Quartile Based)
# ==========================================

q1 = features["Monetary"].quantile(0.25)
q3 = features["Monetary"].quantile(0.75)

print("Q1 Revenue:", q1)
print("Q3 Revenue:", q3)

def revenue_category(revenue):

    if revenue >= q3:
        return "High"

    elif revenue >= q1:
        return "Medium"

    else:
        return "Low"

features["RevenueCategory"] = (
    features["Monetary"]
    .apply(revenue_category)
)

features[
    [
        "Monetary",
        "RevenueCategory"
    ]
].head()

Q1 Revenue: 341.9
Q3 Revenue: 2247.72


,Monetary,RevenueCategory
0,2433.28,High
1,29018.06,High
2,613.08,Medium
3,580987.04,High
4,22950.22,High


In [17]:
# ==========================================
# Step 11 - Executive Priority
# ==========================================

def executive_priority(row):

    if row["RevenueCategory"] == "High" and row["ChurnProbability"] >= 0.60:
        return "Critical"

    elif row["RevenueCategory"] == "High":
        return "Watch"

    elif row["RevenueCategory"] == "Medium" and row["ChurnProbability"] >= 0.60:
        return "High"

    else:
        return "Low"

features["ExecutivePriority"] = features.apply(
    executive_priority,
    axis=1
)

features[
    [
        "RevenueCategory",
        "ChurnProbability",
        "ExecutivePriority"
    ]
].head()

,RevenueCategory,ChurnProbability,ExecutivePriority
0,High,0.224913,Watch
1,High,0.000830,Watch
2,Medium,0.744811,High
3,High,0.000648,Watch
4,High,0.000845,Watch


In [18]:
# ==========================================
# Step 12 - Executive Summary
# ==========================================

executive_summary = (
    features
    .groupby("ExecutivePriority")
    .agg(
        Customers=("Customer ID", "count"),
        TotalRevenue=("Monetary", "sum"),
        RevenueAtRisk=("RevenueAtRisk", "sum")
    )
    .sort_values(
        by="RevenueAtRisk",
        ascending=False
    )
)

executive_summary

,Customers,TotalRevenue,RevenueAtRisk
ExecutivePriority,,,
Watch,1411,1.357565e+07,1.499363e+06
Low,3061,2.172645e+06,9.302073e+05
High,1349,1.023139e+06,7.101229e+05
Critical,60,6.033699e+05,4.165305e+05


In [19]:
# ==========================================
# Step 13 - Top 20 Executive Priority Customers
# ==========================================

top_customers = (
    features
    .sort_values(
        by=["RevenueAtRisk", "ChurnProbability"],
        ascending=[False, False]
    )
)

top_customers[
    [
        "Customer ID",
        "Segment",
        "Monetary",
        "ChurnProbability",
        "RevenueAtRisk",
        "RevenueCategory",
        "ExecutivePriority"
    ]
].head(20)

,Customer ID,Segment,Monetary,ChurnProbability,RevenueAtRisk,RevenueCategory,ExecutivePriority
4894,16446.0,Potential Loyalists,168472.50,0.675477,113799.353790,High,Critical
3513,17450.0,Champions,244784.25,0.271379,66429.208672,High,Watch
747,12346.0,At Risk,77556.46,0.576094,44679.818320,High,Watch
1838,16754.0,At Risk,65500.07,0.601590,39404.181749,High,Critical
2963,12415.0,Champions,144458.37,0.250552,36194.320405,High,Watch
4360,15749.0,At Risk,44534.30,0.703359,31323.609026,High,Critical
5244,14096.0,Champions,65164.79,0.467368,30455.910821,High,Watch
353,17850.0,At Risk,51208.87,0.591284,30278.962947,High,Watch
4977,15098.0,Need Attention,39916.50,0.698835,27895.040750,High,Critical
794,13902.0,At Risk,34095.26,0.690306,23536.146560,High,Critical


In [20]:
# ==========================================
# Step 14 - Save Final Dataset
# ==========================================

top_customers.to_csv(
    "/content/drive/MyDrive/Revenue_Intelligence_Project/Data/revenue_at_risk.csv",
    index=False
)

print("Final Revenue-at-Risk Dataset Saved Successfully!")

Final Revenue-at-Risk Dataset Saved Successfully!
